# Segmentation sweep on KAGGLE (resumable, persistent)

Finishes the U-Net vs DeepLabV3+ sweep. Kaggle advantages over Colab free: up to ~9–12h uninterrupted
sessions, a P100 (faster than T4), and persistent `/kaggle/working/`. All 4 remaining runs should fit in one session.

**Before running, one-time Kaggle setup:**
1. Create a Kaggle account and **verify your phone number** (required to unlock the GPU).
2. New Notebook → right sidebar → **Settings → Accelerator → GPU P100**.
3. Settings → **Internet → ON** (needed to git-clone the code + dataset).
4. Upload your two completed result files as a Kaggle Dataset (see cell 4), OR upload them via the
   notebook's 'Add Data' if you prefer — cell 4 shows both options.

Everything in `src/` is identical to the Colab version — only paths/persistence differ.

## 1. Clone your code repo + install deps

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('SteelDefectX'):
    !git clone https://github.com/d-mondal/SteelDefectX.git
os.chdir('/kaggle/working/SteelDefectX')
!git pull
!pip install -q segmentation-models-pytorch albumentations torchmetrics

## 2. Download the dataset (git clone — the method that works)

In [ ]:
DATA_ROOT = '/kaggle/working/sdx_data'
if not os.path.isdir(f'{DATA_ROOT}/train'):
    !apt-get -qq install git-lfs && git lfs install
    !git clone https://huggingface.co/datasets/Zhaosxian/SteelDefectX {DATA_ROOT}
print('data at:', DATA_ROOT, '| train imgs:', len(os.listdir(f'{DATA_ROOT}/train')))

## 3. Persistent results dir + frozen split

In [ ]:
# /kaggle/working persists as the notebook's output between saved sessions
RESULTS_DIR = '/kaggle/working/results/segmentation'
CKPT_DIR    = '/kaggle/working/models/segmentation'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('data/splits', exist_ok=True)
!python -m src.data.make_split \
    --train-text {DATA_ROOT}/train-text.json \
    --out-dir data/splits --val-frac 0.15 --seed 42

## 4. Restore your completed seed0/seed1 results

**Two ways — pick one:**

**(a) Easiest — upload via 'Add Data':** In the right sidebar, click **+ Add Input → Upload**, upload your
two `*_test_metrics.json` files as a new dataset. They'll appear under `/kaggle/input/<your-dataset-name>/`.
Then set `UPLOAD_DIR` below to that path and run the cell.

**(b) Or skip restoring** and just let Kaggle re-run seed0/seed1 too — you have the GPU hours, and it makes
the whole sweep come from one consistent environment (arguably cleaner for the report). If you choose this,
set `UPLOAD_DIR = None`.

In [ ]:
import shutil, glob
UPLOAD_DIR = None   # e.g. '/kaggle/input/steeldefectx-seed-results'  (option a), or None (option b)

if UPLOAD_DIR:
    for f in glob.glob(f'{UPLOAD_DIR}/*_test_metrics.json'):
        shutil.copy(f, RESULTS_DIR)
        print('restored ->', os.path.basename(f))
else:
    print('no restore -> all seeds will run fresh on Kaggle (fine; you have the hours)')
print('results now present:', [os.path.basename(f) for f in glob.glob(f'{RESULTS_DIR}/*.json')])

## 5. Configure

In [ ]:
from src.config import Config
cfg = Config(
    data_root=DATA_ROOT,
    split_dir='data/splits',
    encoder='resnet34',
    batch_size=16,
    epochs=40,
    seeds=(0, 1, 2),
    out_dir=RESULTS_DIR,
    ckpt_dir=CKPT_DIR,
    use_wandb=False,
)
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
done = {os.path.basename(f).replace('_test_metrics.json','') for f in glob.glob(f'{RESULTS_DIR}/*_test_metrics.json')}
todo = [f'{a}_{cfg.encoder}_seed{s}' for a in ('unet','deeplabv3plus') for s in cfg.seeds if f'{a}_{cfg.encoder}_seed{s}' not in done]
print('already done:', sorted(done) or 'none')
print('still to run:', todo or 'ALL DONE')

## 6. Run the sweep (skips completed runs)

**Tip:** use **Save Version → Save & Run All (Commit)** to run this in the background — it keeps going even
if you close the tab, and saves all of `/kaggle/working` as output when done. That's the disconnect-proof way.

In [ ]:
from src.segmentation.train import run_comparison
results = run_comparison(cfg)

## 7. Per-class IoU across all runs

In [ ]:
import json
for f in sorted(glob.glob(f'{RESULTS_DIR}/*_test_metrics.json')):
    m = json.load(open(f))
    print(f"\n{os.path.basename(f)}  overall IoU={m['IoU']}")
    pc = sorted(m['per_class'].items(), key=lambda x: x[1]['IoU'])
    for c, v in pc[:5]:
        print(f"   worst: {c:32s} IoU={v['IoU']*100:5.1f}  (n={v['n']})")

## 8. FINAL publication-ready tables (head-to-head + per-class)

Run this any time — works with a partial sweep and fills in as runs complete. When all 6 are done this is
your thesis table: per-arch mean±std IoU/F1max/AUROC vs the paper's 37.49, plus per-class IoU across seeds.

In [ ]:
from src.evaluation.aggregate import aggregate
agg = aggregate(RESULTS_DIR)